## Working with TiTiler-EoPF - Mosaic

This notebook demonstrates how to use the TiTiler-EoPF service to visualize a **collection** (STAC) of EOPF Zarr datastores.


In [ ]:
# Start titiler-eopf services locally with Docker

!docker compose up api -d

In [ ]:
import json
import httpx2 as httpx
from folium import Map, TileLayer

%matplotlib inline

In [ ]:
titiler_endpoint = "http://127.0.0.1:8000"

In [ ]:
r = httpx.get(f"{titiler_endpoint}/_mgmt/health")
print(r.json())

### Conformances

In [ ]:
r = httpx.get(f"{titiler_endpoint}/conformance").json()
print(json.dumps(r, indent=4))

In [ ]:
collection_id = "sentinel-2-l2a"
bbox = [11.393460776835221, 41.42010137184542, 12.466572328262494, 42.426911995973605]
date = "2026-01-17/2026-01-18"

## Collection Info

In [ ]:
# Fetch Metadata for all available variables
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/info",
    params={
        "bbox": "11.393460776835221,41.42010137184542,12.466572328262494,42.426911995973605",
        "datetime": "2026-08-01/2026-08-06",
    },
    timeout=20,
).json()

print(json.dumps(r, indent=4))

## Display tiles

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/WebMercatorQuad/tilejson.json",
    params={
        "bbox": "11.393460776835221,41.42010137184542,12.466572328262494,42.426911995973605",
        "datetime": "2026-08-01/2026-08-06",
        # "ids": "S2B_MSIL2A_20260731T100559_N0512_R022_T33TTG_20260731T142842",
        "assets": "reflectance|bands=red,green,blue",
        "rescale": "0,1",
        "tilesize": 256,
    },
    timeout=10,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=8
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)
m

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/WebMercatorQuad/tilejson.json",
    params={
        "bbox": "11.393460776835221,41.42010137184542,12.466572328262494,42.426911995973605",
        "datetime": "2026-08-01/2026-08-06",
        # "ids": "S2B_MSIL2A_20260731T100559_N0512_R022_T33TTG_20260731T142842",
        "assets": "reflectance|bands=red,nir",
        "expression": "(b2-b1)/(b2+b1)",
        "rescale": "-1,1",
        "colormap_name": "viridis",
        "tilesize": 256,
    },
    timeout=10,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=8
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)
m